# 11 — Multi-Agent Patterns

## Learning requirements
Không dùng multi-agent chỉ vì task “nghe phức tạp”.

Học 5 pattern:
1. subagents/supervisor;
2. router;
3. handoff;
4. skills/context loading;
5. custom LangGraph workflow.

Single agent + đúng tools/context thường đơn giản, rẻ và dễ evaluate hơn.

## Architecture selection

### Router
One classification/decomposition step dispatches work to specialist(s).

### Supervisor
Main agent giữ conversation context và gọi subagents như tools.

### Handoff
Active conversational responsibility chuyển sang specialist.

### Parallel workers
Fan-out independent tasks rồi aggregate.

Đánh đổi chính:
- context isolation;
- latency;
- duplicated tokens;
- coordination errors;
- observability/evaluation complexity.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from pathlib import Path
import sys
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))
from src.providers import get_chat_model

model = get_chat_model()

research_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a research specialist. Return concise evidence and uncertainties.",
)

@tool("research", description="Delegate a bounded research question to the research specialist.")
def call_research_agent(query: str) -> str:
    result = research_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content

supervisor = create_agent(
    model=model,
    tools=[call_research_agent],
    system_prompt="You are the supervisor. Delegate research only when specialization is useful.",
)

## Exercise

Implement cùng use case bằng:
- single agent;
- router;
- supervisor + 2 specialists.

Dataset ít nhất 20 prompts.

Measure:
- success rate;
- routing accuracy;
- average model calls;
- latency;
- token usage;
- coordination failure.

## Required output
`multi_agent_system/` design với:
- router/supervisor;
- research specialist;
- code specialist;
- aggregator or final synthesis;
- context-sharing rules.

## Done criteria
Bạn có evidence multi-agent giúp use case; nếu không, chọn single-agent.